In [1]:
import os

In [2]:
os.makedirs("Neuroselectivity", exist_ok=True)

In [3]:
with open("Neuroselectivity/__init__.py", "w") as f:
    f.write("# Neuroselectivity package\n")

In [25]:
anova_code = """
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
from scipy.stats import f

    
def run_ANOVA(binned_data, label_col):  #, include_site_info=True):

    # Identify time columns
    time_cols = [c for c in binned_data.columns if c.startswith("time")]
    
    # Melt into long format
    long_data = binned_data.melt(
        id_vars=['siteID', label_col],
        value_vars=time_cols,
        var_name='time_period',
        value_name='activity'
    )
    
    
    long_data = long_data.rename(columns={label_col: "labels"})
    
    df1 = (long_data.groupby("siteID")[["labels"]].nunique() - 1).rename(columns = {"labels":"df1"})
    long_data = long_data.merge(df1.reset_index())
    
    
    df2 = long_data.groupby(["siteID", "time_period"]).agg(df2 = ("labels", "count"))
    long_data = long_data.merge(df2.reset_index())
    long_data["df2"] = long_data["df2"] - long_data["df1"] - 1
    
    
    group_means = long_data.groupby(["siteID", "time_period", "labels"]).agg(group_mean = ("activity", "mean"))
    long_data = long_data.merge(group_means.reset_index())
    
    overall_means = long_data.groupby(["siteID", "time_period"]).agg(overall_mean = ("activity", "mean"))
    long_data = long_data.merge(overall_means.reset_index())
    
    long_data["group_deviation_squared"] = (long_data.group_mean - long_data.overall_mean)**2
    long_data["residual_squared"] = (long_data.group_mean - long_data.activity)**2
    
    anova_results = long_data.groupby(["siteID", "time_period"]).agg(
    df1 = ("df1", "mean"),
    df2 = ("df2", "mean"),
    SS_group = ("group_deviation_squared", "sum"),
    SS_residual = ("residual_squared", "sum"))

    anova_results["F_stat"] = (1/anova_results.df1 * anova_results.SS_group)/(1/anova_results.df2 * anova_results.SS_residual)    
    anova_results["p_val"] = 1 - f.cdf(anova_results.F_stat, anova_results.df1, anova_results.df2)
    
    return anova_results.reset_index()

def get_time_bin_center(time_bins):

    avg_times = (
        time_bins.str.extract(r'time\.(-?\d+)_(-?\d+)')   # extract the two groups of digits
            .astype(int)                         # convert to integers
            .mean(axis=1)                        # compute the average row-wise
    )

    return avg_times

def plot_ANOVA_results(anova_results, alpha=0.01, baseline_time=500, ax=None):
    # If no axis is provided, create one
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    anova_results2 = anova_results.reset_index()
    anova_results2["stat_sig"] = anova_results2.p_val < alpha 
    
    # Assumes get_time_bin_center is defined in your environment
    anova_results2["time"] = ns.get_time_bin_center(anova_results2["time_period"])
    
    num_selective_results = anova_results2.groupby("time").agg(num_selective=("stat_sig", "sum")).reset_index()
    
    # Calculate percentage
    num_sites = len(np.unique(anova_results.reset_index().siteID))
    y_values = 100 * num_selective_results.num_selective / num_sites

    # Stylistic Plotting
    ax.plot(num_selective_results.time, y_values, color='black', linewidth=3, label='ANOVA Selectivity')
    
    # Reference Lines (Matching AUROC style)
    ax.axhline(alpha * 100, color='red', linestyle=':', alpha=0.5, label='Chance (Alpha)')
    ax.axvline(baseline_time, color='gray', linestyle='--', alpha=0.5)

    # Formatting
    ax.set_title('ANOVA Population Selectivity', fontsize=14)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('% Neurons Selective')
    ax.grid(True, alpha=0.2)
    return ax
    

"""

with open("Neuroselectivity/anova.py", "w") as f:
    f.write(anova_code)

In [5]:
%%writefile Neuroselectivity/preprocessing.py
import os
import glob
import numpy as np
import pandas as pd
from scipy.io import loadmat

def load_mat_folder(folder_path):
    
    def extract_array(val, n_rows):
        if val is None:
            return np.full(n_rows, np.nan)
        val = np.array(val).flatten()
        if len(val) == 0:
            return np.full(n_rows, np.nan)
        elif len(val) == 1:
            return np.repeat(val[0], n_rows)
        elif len(val) == n_rows:
            return val
        else:
            return np.repeat(val[0], n_rows)

    def unpack_mat_struct(mat_struct, n_rows):
        result = {}
        if isinstance(mat_struct, dict):
            for k, v in mat_struct.items():
                if isinstance(v, (dict, np.void)):
                    sub = unpack_mat_struct(v, n_rows)
                    for subk, subv in sub.items():
                        result[f"{k}_{subk}"] = subv
                else:
                    result[k] = extract_array(v, n_rows)
        elif isinstance(mat_struct, np.void):
            for k in mat_struct.dtype.names:
                v = mat_struct[k]
                if isinstance(v, (dict, np.void)):
                    sub = unpack_mat_struct(v, n_rows)
                    for subk, subv in sub.items():
                        result[f"{k}_{subk}"] = subv
                else:
                    result[k] = extract_array(v, n_rows)
        return result

    all_dfs = []

    mat_files = glob.glob(os.path.join(folder_path, "*.mat"))

    for file in mat_files:
        data = loadmat(file, simplify_cells=True)
        
        # Raster data
        raster_data = data.get("raster_data", None)
        if raster_data is None:
            continue
        df = pd.DataFrame(raster_data)
        n_rows = len(df)
        
        # Raster labels
        raster_labels = data.get("raster_labels", {})
        flat_labels = unpack_mat_struct(raster_labels, n_rows)
        for k, v in flat_labels.items():
            df[k] = v
        
        # Raster site info
        raster_site_info = data.get("raster_site_info", {})
        flat_site = unpack_mat_struct(raster_site_info, n_rows)
        for k, v in flat_site.items():
            df[k] = v
        
        # Timing info
        timing_info = data.get("timing_info", {})
        flat_timing = unpack_mat_struct(timing_info, n_rows)
        for k, v in flat_timing.items():
            df[k] = v
        
        # Source file
        name = os.path.basename(file).replace(".mat", "")
        df["source_file"] = name
        
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.set_index("source_file")
    
    return combined_df


def bin_spikes_only(df, bin_size, sampling_interval):
    # Identify spike columns (assumes numeric column names for spikes)
    spike_cols = [col for col in df.columns if str(col).isdigit()]
    spike_cols = sorted(spike_cols, key=int)
    
    # Identify metadata columns (everything else)
    metadata_cols = [col for col in df.columns if col not in spike_cols]
    
    # Bin the spike columns
    binned_dict = {}
    for start in range(0, len(spike_cols), sampling_interval):
        end = start + bin_size
        bin_cols = spike_cols[start:end]
        if len(bin_cols) == 0:
            continue
        bin_name = f"time_bin_{start}_{start+len(bin_cols)-1}"
        binned_dict[bin_name] = df[bin_cols].mean(axis=1)
    
    # Create DataFrame for binned spikes
    binned_spikes = pd.DataFrame(binned_dict, index=df.index)
    
    # Concatenate metadata columns back
    final_df = pd.concat([df[metadata_cols], binned_spikes], axis=1)
    
    return final_df


def rename_columns(df, time_unit=10):
    site_info_cols = [
        "original_file_name", "experiment_type_name", "training_phase_name",
        "brain_area_name", "experiment_type", "feature_set_position",
        "feature_set_pairing_letter", "feature_set_pairing_number",
        "training_phase_number", "brain_area_number", "save_file_name"
    ]
    
    new_cols = {}
    for col in df.columns:
        if col.startswith("time_bin_"):
            # Extract start and end numbers
            parts = col.replace("time_bin_", "").split("_")
            start = int(parts[0])
            end = start + time_unit  # add unit to end
            new_col = f"time.{start}_{end}"
        elif col in site_info_cols:
            new_col = f"site_info.{col}"
        else:
            new_col = f"labels.{col}"
        
        new_cols[col] = new_col
    
    return df.rename(columns=new_cols)


Overwriting Neuroselectivity/preprocessing.py


In [1]:
AUROC_code = """

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import re


def analyze_dataframe_selectivity_corrected(df, n_shuffles=500):
    time_cols = [c for c in df.columns if c.startswith('time.')]
    stim_col = 'labels.stimulus_ID'
    neuron_col = 'site_info.source_file'
    
    neurons = df[neuron_col].unique()
    stims = df[stim_col].unique()
    n_stim = len(stims)
    
    alpha_corrected = 0.05 / n_stim
    lower_percentile = (alpha_corrected / 2) * 100
    upper_percentile = (1 - (alpha_corrected / 2)) * 100
    
    sig_matrix = np.zeros((n_stim, len(neurons), len(time_cols)))

    for n_idx, neuron_id in enumerate(neurons):
        neuron_df = df[df[neuron_col] == neuron_id]
        
        for s_idx, stim in enumerate(stims):
            data_stim = neuron_df[neuron_df[stim_col] == stim][time_cols].values
            data_others = neuron_df[neuron_df[stim_col] != stim][time_cols].values
            
            if len(data_stim) < 3 or len(data_others) < 3: continue

            obs_aurocs = compute_fast_auroc_vectorized(data_stim, data_others)
            
            combined_data = np.vstack([data_stim, data_others])
            n_s = len(data_stim)
            shuff_results = []
            
            for _ in range(n_shuffles):
                perm_idx = np.random.permutation(len(combined_data))
                shuff_aurocs = compute_fast_auroc_vectorized(combined_data[perm_idx[:n_s]], 
                                                             combined_data[perm_idx[n_s:]])
                shuff_results.append(shuff_aurocs)
            
            shuff_results = np.array(shuff_results)
            low_tail = np.percentile(shuff_results, lower_percentile, axis=0)
            high_tail = np.percentile(shuff_results, upper_percentile, axis=0)
            
            sig_matrix[s_idx, n_idx, :] = (obs_aurocs <= low_tail) | (obs_aurocs >= high_tail)

    per_stim_curves = np.mean(sig_matrix, axis=1) * 100
    overall_curve = np.mean(np.any(sig_matrix, axis=0), axis=0) * 100
    
    return time_cols, stims, per_stim_curves, overall_curve

def plot_auroc_results(time_cols, stim_names, per_stim_curves, overall_curve, 
                       baseline_time=500, correction='none', plot_mode='all'):
    #Args: correction: 'none' (raw %), 'global' (overall baseline), 'per_stim' (individual baselines), plot_mode: 'all', 'main' (only black line), 'individual' (only stimulus lines)
    
    # Clean up the time labels
    time_points = []
    for col in time_cols:
        match = re.search(r'\d+', col)
        time_points.append(int(match.group()) if match else 0)
    time_points = np.array(time_points)
    
    plt.figure(figsize=(10, 6))
    mask = time_points < baseline_time

    # 1. Plot Individual Curves
    if plot_mode in ['all', 'individual']:
        for i, stim_name in enumerate(stim_names):
            curve = per_stim_curves[i]
            if correction == 'per_stim':
                curve = curve - np.mean(curve[mask])
            elif correction == 'global':
                curve = curve - np.mean(overall_curve[mask])
            
            plt.plot(time_points, curve, label=f'Selectivity: {stim_name}', 
                     alpha=0.6, linestyle='--')

    # 2. Plot Overall Curve
    if plot_mode in ['all', 'main']:
        curve = overall_curve
        if correction != 'none':
            curve = curve - np.mean(overall_curve[mask])
        
        plt.plot(time_points, curve, color='black', linewidth=3, label='Overall Selectivity')

    # 3. Reference Lines
    ref_val = 0 if correction != 'none' else 5
    plt.axhline(ref_val, color='red', linestyle=':', alpha=0.5, label='Baseline/Chance')
    plt.axvline(baseline_time, color='gray', linestyle='--', alpha=0.5)

    # Formatting
    title_suffix = f" ({correction} correction)" if correction != 'none' else ""
    plt.title(f'Neural Population Selectivity{title_suffix}', fontsize=14)
    plt.xlabel('Time (ms)')
    plt.ylabel('% Neurons Selective' if correction == 'none' else 'Δ % Selective')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    
"""
with open("Neuroselectivity/AUROC.py", "w") as f:
    f.write(AUROC_code)

In [3]:
simple_index_code = """

from scipy.stats import f_oneway, f, ttest_ind
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd

def calculate_si_logic(mean1, mean2, b_val=0, corrected=True):
    # Internal math unit to handle the SI calculation.
    if corrected:
        # Subtract scalar baseline from both time-resolved means
        Rf = mean1 - b_val
        Rn = mean2 - b_val
        
        denom = Rf + Rn
        # Use epsilon to avoid 0/0
        si = (Rf - Rn) / np.where(denom == 0, 1e-9, denom)
        
        # Apply Tsao 'Opposite Sign' logic (Vectorized)
        si = np.where((Rf > 0) & (Rn < 0), 1.0, si)
        si = np.where((Rf < 0) & (Rn > 0), -1.0, si)
    else:
        # Raw Method (Old)
        denom = mean1 + mean2
        si = (mean1 - mean2) / np.where(denom == 0, 1e-9, denom)
        
    return np.clip(si, -1, 1)

def compute_population_si(
    df, 
    cond1_col="labels.sample_stimulus_category", 
    cond1_val="cat", 
    cond2_val="dog",
    neuron_col="labels.neuron_number", 
    spike_cols_prefix="time.",
    corrected=True,
    baseline_window=(0, 50)
):
    # 1. Identify valid time columns
    time_cols = [c for c in df.columns if c.startswith(spike_cols_prefix) and c != 'time.6000_6250']
    neurons = df[neuron_col].unique()
    
    results = {}

    for neuron in neurons:
        n_df = df[df[neuron_col] == neuron]
        
        # 2. Split into the two condition groups
        group1 = n_df[n_df[cond1_col] == cond1_val][time_cols]
        group2 = n_df[n_df[cond1_col] == cond2_val][time_cols]
        
        if group1.empty or group2.empty:
            continue
            
        # 3. Calculate Means (Hz)
        mean1_hz = group1.mean() * 1000
        mean2_hz = group2.mean() * 1000
        
        # 4. Optional Baseline
        b_val = 0
        if corrected:
            b_cols = time_cols[baseline_window[0]:baseline_window[1]]
            b_val = n_df[b_cols].mean().mean() * 1000 
        
        # 5. Logic unit
        si_timecourse = calculate_si_logic(
            mean1_hz, 
            mean2_hz, 
            b_val=b_val, 
            corrected=corrected
        )
        results[neuron] = si_timecourse

    final_df = pd.DataFrame.from_dict(results, orient='index', columns=time_cols)
    final_df.index.name = neuron_col
    return final_df

def compute_one_vs_all_selectivity(
    df, 
    neuron_col="labels.neuron_number", 
    stim_col="labels.sample_stimulus_category", 
    spike_cols_prefix="time.",
    corrected=True,
    baseline_window=(0, 50)):
    
    time_cols = sorted(
        [c for c in df.columns if c.startswith(spike_cols_prefix) and c != 'time.6000_6250'],
        key=extract_time_numeric
    )
    unique_stimuli = df[stim_col].unique()
    one_vs_all_sis = {}

    for stim in unique_stimuli:
        si_dict = {}
        for neuron in df[neuron_col].unique():
            n_df = df[df[neuron_col] == neuron]
            group1 = n_df[n_df[stim_col] == stim][time_cols]
            group2 = n_df[n_df[stim_col] != stim][time_cols]
            
            if group1.empty or group2.empty:
                continue
            
            mean1_hz = group1.mean() * 1000
            mean2_hz = group2.mean() * 1000
            
            b_val = 0
            if corrected:
                b_cols = time_cols[baseline_window[0]:baseline_window[1]]
                b_val = n_df[b_cols].mean().mean() * 1000
            
            si_series = calculate_si_logic(
                mean1_hz, 
                mean2_hz, 
                b_val=b_val, 
                corrected=corrected
            )
            si_dict[neuron] = si_series

        si_df = pd.DataFrame.from_dict(si_dict, orient='index', columns=time_cols)
        si_df.index.name = neuron_col
        one_vs_all_sis[(stim, "all_others")] = si_df
    
    return one_vs_all_sis

def extract_time_numeric(col_name):
    if isinstance(col_name, (int, float)):
        return col_name
    match = re.search(r'(\d+)', str(col_name))
    if match:
        return int(match.group(1))
    return None

def compute_baseline_thresholds(one_vs_all_sis, pre_stimulus_end=50, k=2, method='corrected'):
    if isinstance(one_vs_all_sis, pd.DataFrame):
        one_vs_all_sis = {'single': one_vs_all_sis}

    neurons = set()
    for df in one_vs_all_sis.values():
        neurons.update(df.index)
    
    baseline_samples = {n: [] for n in neurons}
    
    for stim, df in one_vs_all_sis.items():
        pre_cols = [c for c in df.columns if (t := extract_time_numeric(c)) is not None and t < pre_stimulus_end]
        for neuron in df.index:
            vals = df.loc[neuron, pre_cols].values
            baseline_samples[neuron].extend(vals)
    
    threshold_dict = {}
    for neuron, samples in baseline_samples.items():
        v = pd.Series(samples).dropna()
        if v.empty:
            threshold_dict[neuron] = np.inf
            continue
        if method == 'raw':
            threshold_dict[neuron] = v.mean() + (k * v.std())
        elif method == 'corrected':
            threshold_dict[neuron] = k * v.std()
            
    thresh_series = pd.Series(threshold_dict)
    thresh_series.index.name = list(one_vs_all_sis.values())[0].index.name
    return thresh_series

def compute_integrated_selectivity(one_vs_all_sis, thresholds):
    if isinstance(one_vs_all_sis, pd.DataFrame):
        one_vs_all_sis = {'single_comparison': one_vs_all_sis}

    per_stim_records = []
    all_masks = []

    for stim, df_si in one_vs_all_sis.items():
        mask = df_si.gt(thresholds, axis=0)
        all_masks.append(mask)
        percents = mask.mean() * 100
        for time_bin, p in percents.items():
            per_stim_records.append({
                "stimulus": stim, 
                "time_bin": time_bin, 
                "percent_selective": p
            })
    
    per_stim_df = pd.DataFrame(per_stim_records)
    per_stim_df["time_numeric"] = per_stim_df["time_bin"].apply(extract_time_numeric)

    if len(all_masks) > 1:
        combined_mask = pd.concat(all_masks).groupby(level=0).any()
        overall_percents = combined_mask.mean() * 100
    else:
        overall_percents = all_masks[0].mean() * 100
    
    overall_df = pd.DataFrame({
        "time_bin": overall_percents.index,
        "percent_selective": overall_percents.values
    })
    overall_df["time_numeric"] = overall_df["time_bin"].apply(extract_time_numeric)
    
    return per_stim_df.sort_values("time_numeric"), overall_df.sort_values("time_numeric")

def plot_integrated_selectivity2(per_stim_df, overall_df, mode="overall", target_stimulus=None, chance_level=5, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    # --- 1. PLOT INDIVIDUAL STIMULI LINES ---
    if mode in ["all", "single", "both"] and per_stim_df is not None:
        plot_stim_df = per_stim_df.copy()
        
        if mode == "single":
            # Bulletproof filter: search for target characters within the tuple/string
            target_str = str(target_stimulus)
            mask = plot_stim_df['stimulus'].astype(str).str.contains(target_str, case=False, na=False)
            plot_df = plot_stim_df[mask]
            
            if not plot_df.empty:
                sns.lineplot(
                    data=plot_df, x="time_numeric", y="percent_selective", 
                    color="black", linewidth=2, marker=None, errorbar=None, 
                    label=f"Target: {target_stimulus}", ax=ax
                )
            else:
                print(f"Warning: No data found for {target_str}")
        
        else:
            # For 'all' or 'both', keep colors to distinguish between them
            plot_stim_df['stimulus'] = plot_stim_df['stimulus'].astype(str)
            sns.lineplot(
                data=plot_stim_df, x="time_numeric", y="percent_selective", 
                hue="stimulus", palette="tab10", marker=None, errorbar=None, 
                alpha=0.4 if mode == "both" else 1.0, ax=ax
            )

    # --- 2. PLOT OVERALL POPULATION LINE ---
    if mode in ["overall", "both"] and overall_df is not None:
        sns.lineplot(
            data=overall_df, x="time_numeric", y="percent_selective", 
            color="black", linewidth=3, marker=None, errorbar=None, 
            label="Overall Population", ax=ax
        )

    # --- 3. REFERENCE LINES & FORMATTING ---
    if chance_level is not None:
        ax.axhline(y=chance_level, color='red', linestyle=':', alpha=0.5, label=f"Chance ({chance_level}%)")
    
    # Gray onset line at 50ms
    ax.axvline(50, color='gray', linestyle='--', alpha=0.5)

    ax.set_title(f"SI Population Selectivity: Raw", fontsize=14)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Percent of neurons selective (%)")
    ax.set_ylim(0, 60)
    
    # Check for labels before adding legend
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(fontsize='small', frameon=False, loc='upper right')
    
    ax.grid(True, alpha=0.2)
    
    return ax

def calculate_si_comparison_timecourse(df):
    time_cols = [c for c in df.columns if c.startswith("time.")]
    baseline_cols = time_cols[:10] 
    neurons = df["source_file"].unique()
    corrected_results = []

    for neuron in neurons:
        n_df = df[df["source_file"] == neuron]
        face_mask = n_df["labels.is_face"] == True
        obj_mask = n_df["labels.is_face"] == False
        if not face_mask.any() or not obj_mask.any(): continue
            
        r_faces_t = n_df[face_mask][time_cols].mean() * 1000
        r_objs_t = n_df[obj_mask][time_cols].mean() * 1000
        
        b_mean = n_df[baseline_cols].mean().mean() * 1000
        Rf = r_faces_t - b_mean
        Rn = r_objs_t - b_mean
        
        fsi_corr = (Rf - Rn) / (Rf + Rn)
        fsi_corr[(Rf > 0) & (Rn < 0)] = 1.0
        fsi_corr[(Rf < 0) & (Rn > 0)] = -1.0
        corrected_results.append(np.clip(fsi_corr, -1, 1))
        
    return pd.DataFrame(corrected_results, columns=time_cols)

def plot_si_integrated_style(df_si, threshold=0.333, chance_level=5, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    is_selective = df_si > threshold
    percent_selective = is_selective.mean() * 100
    time_numeric = [int(c.split('.')[1].split('_')[0]) for c in df_si.columns]
    
    plot_df = pd.DataFrame({'time_numeric': time_numeric, 'percent_selective': percent_selective.values})

    sns.lineplot(data=plot_df, x="time_numeric", y="percent_selective", color="black", linewidth=3, ax=ax, label=f"FSI > {threshold}")

    if chance_level is not None:
        ax.axhline(y=chance_level, color='red', linestyle=':', alpha=0.5, label=f"Chance ({chance_level}%)")
    ax.axvline(50, color='gray', linestyle='--', alpha=0.5)

    ax.set_title(f"SI Population Selectivity: Tsao Corrected (FSI > {threshold})")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Percent of neurons selective (%)")
    ax.set_ylim(0, 100)
    ax.legend(fontsize='small', frameon=False)
    ax.grid(True, alpha=0.2)
    return ax


"""

with open("Neuroselectivity/SI.py", "w") as f:
    f.write(simple_index_code)

In [5]:
%%writefile Neuroselectivity/__init__.py

from .preprocessing import *
from .anova import *
from .AUROC import *
from .SI import *

__all__ = [
    "bin_spikes_only",
    "run_safe_ANOVA",
    "load_mat_folder",
    "rename_columns",
    "plot_percent_selective"
]

Overwriting Neuroselectivity/__init__.py
